# Experiment run and results

This notebook records the completed major experiments in the project. The saved outputs
are visible when opened in Jupyter or a notebook viewer. **Run All** in the default
configuration regenerates the tables from the versioned aggregate JSON under
[`results/`](../results/). It works offline from a checkout with a Python Jupyter kernel,
including when the local models, datasets, Spark, DuckDB or llama.cpp are unavailable.

The displayed metrics are from the original completed runs. A separate opt-in section at
the end calls the project runners if you want to reproduce the experiments. Source commits
are listed below and the JSON exports retain the full input and source hashes. Private
article text, predictions, weights and reviewer records are kept out of the notebook.

These are native public-data studies and a teacher-labelled entity pilot. The official
SemEval Task 10 annotations and a verified organizer scoring setup were unavailable.

In [1]:
from html import escape
import json
from pathlib import Path

from IPython.display import display

cwd = Path.cwd()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "results" / "experiments.json").is_file()
             and (p / "experiments" / "reproduce.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Start the kernel in the repository root or its experiments/ directory.")

def saved(name):
    with (root / "results" / f"{name}.json").open(encoding="utf-8") as stream:
        return json.load(stream)

def table(headers, rows):
    rows = [[str(item) for item in row] for row in rows]
    assert rows and all(len(row) == len(headers) for row in rows)
    header = "".join(f"<th>{escape(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(v)}</td>" for v in row) + "</tr>"
                   for row in rows)
    plain = "\n".join(" | ".join(row) for row in [list(headers), *rows])
    display({"text/html": f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>",
             "text/plain": plain}, raw=True)

def score(value):
    return "—" if value is None else f"{value:.4f}"

def percent(value):
    return f"{value:.1%}"

catalog = saved("experiments")["experiments"]
def study(name):
    runs = [run for run in catalog if run["study"] == name]
    if not runs:
        raise KeyError(f"Study missing from results/experiments.json: {name}")
    return runs

def classification(runs):
    table(("Model or control", "Test rows", "Macro F1", "Seed SD", "Seeds"), (
        (r["name"], r["n"], score(r["value"]), score(r.get("seed_std")),
         ", ".join(map(str, r.get("seeds") or [])) or "—")
        for r in runs
    ))

print(f"Loaded {len(catalog)} result records from results/experiments.json.")

Loaded 121 result records from results/experiments.json.


## 1. Prepared data and Spark SQL audit

The main audit covers the prepared CARDS and ProBel train, development and test
splits. The native multilingual extension has a separate audit. The overlap checks
apply to IDs and normalized exact text within each dataset. The timing is local.

In [2]:
spark = saved("spark-audit")
multilingual = saved("multilingual")
audit_rows = [("CARDS and ProBel", spark), ("Native multilingual", multilingual["spark_audit"])]
table(("Audit", "Records", "Invalid records", "Cross-split overlap groups", "Passed"), (
    (name, f'{audit["records"]:,}', audit["invalid_records"],
     sum(audit["cross_split_overlap_groups"].values()), audit["passed"])
    for name, audit in audit_rows
))
table(("Dataset", "Split", "Language", "Rows"), (
    (r["dataset"], r["split"], r["language"], r["records"])
    for _, audit in audit_rows for r in audit["counts"]
))
print(f'Main audit: Spark {spark["spark_version"]} ({spark["master"]}), '
      f'{spark["startup_seconds"]:.2f} s JVM startup, {spark["audit_seconds"]:.2f} s SQL audit; '
      f'{sum(spark["within_split_duplicate_groups"].values())} within-split duplicate groups.')

Main audit: Spark 4.2.0 (local[2]), 21.86 s JVM startup, 21.85 s SQL audit; 0 within-split duplicate groups.


Audit,Records,Invalid records,Cross-split overlap groups,Passed
CARDS and ProBel,"12,297",0,0,True
Native multilingual,"2,466",0,0,True


Dataset,Split,Language,Rows
cards,dev,en,1000
cards,test,en,2897
cards,train,en,6000
probel,dev,ar,200
probel,dev,en,200
probel,test,ar,200
probel,test,en,200
probel,train,ar,800
probel,train,en,800
hindi,dev,hi,100


## 2. Subtask 1 · Exact-mention entity framing

The model receives the supplied mention and offsets. The test labels come from the same
teacher that annotated the training and development data. The context-only control changes
several feature views together. These scores describe silver-label agreement on seven
mentions, with no Bulgarian or Russian test examples.

In [3]:
entity = saved("entity-framing")
print(f'{entity["teacher"]["name"]}: {entity["candidate_passages"]} candidates, '
      f'{sum(entity["accepted"].values())} accepted, {entity["rejected"]} rejected, '
      f'{entity["abstained"]} abstentions.')
table(("Split", "Accepted", "Language counts"), (
    (split, count, ", ".join(f"{lang}={n}" for lang, n in entity["language_counts"][split].items()))
    for split, count in entity["accepted"].items()
))
table(("Model", "Test mentions", "Fine exact match", "Parent agreement", "Fine micro F1", "Fine macro F1"), (
    (name, m["count"], percent(m["exact_match"]), percent(m["main_role_accuracy"]),
     score(m["micro_f1"]), score(m["macro_f1"]))
    for name, result in entity["silver_agreement"].items() for m in [result["metrics"]]
))

Qwen2.5-3B-Instruct: 70 candidates, 42 accepted, 17 rejected, 11 abstentions.


Split,Accepted,Language counts
train,27,"en=5, bg=6, ru=3, hi=5, pt=8"
dev,8,"en=3, bg=1, ru=1, pt=3"
test,7,"en=2, hi=2, pt=3"


Model,Test mentions,Fine exact match,Parent agreement,Fine micro F1,Fine macro F1
full,7,14.3%,28.6%,0.1333,0.0114
unmarked,7,14.3%,28.6%,0.1429,0.0114


## 3. Subtask 2 · Native classification, graph controls and EMM transfer

The main comparisons use fixed-inventory fine-label macro F1. Graph rows summarize
three seeds with their sample standard deviation. CARDS uses one claim label per
paragraph. ProBel allows multiple techniques or an empty set. EMM reuses fitted
CARDS models without target tuning.

In [4]:
main_studies = ("CARDS · extended hierarchy study", "ProBel · extended bilingual study",
                "CARDS → EMM source transfer")
for name in main_studies:
    print(name)
    classification(study(name))

CARDS · extended hierarchy study
ProBel · extended bilingual study
CARDS → EMM source transfer


Model or control,Test rows,Macro F1,Seed SD,Seeds
MiniLM + hierarchical GCN,2897,0.4682,0.0132,"13, 42, 87"
Character TF-IDF + logistic regression,2897,0.4330,—,42
LSA + logistic regression,2897,0.5510,—,42
MiniLM + hierarchical GCN · random-labels,2897,0.4431,0.0198,"13, 42, 87"
MiniLM + hierarchical GCN · self-loops,2897,0.4582,0.0211,"13, 42, 87"
MiniLM + hierarchical GCN · shuffled-edges,2897,0.4313,0.0232,"13, 42, 87"
MiniLM + logistic regression,2897,0.5274,—,42


Model or control,Test rows,Macro F1,Seed SD,Seeds
MiniLM + hierarchical GCN,400,0.0862,0.0092,"13, 42, 87"
XLM-R + hierarchical GCN,400,0.0983,0.0013,"13, 42, 87"
Character TF-IDF + logistic regression,400,0.0558,—,42
LSA + logistic regression,400,0.0911,—,42
MiniLM + hierarchical GCN · random-labels,400,0.0892,0.0103,"13, 42, 87"
MiniLM + hierarchical GCN · self-loops,400,0.0995,0.0201,"13, 42, 87"
MiniLM + hierarchical GCN · shuffled-edges,400,0.0817,0.0029,"13, 42, 87"
MiniLM + logistic regression,400,0.0876,—,42
XLM-R + logistic regression,400,0.0422,—,42


Model or control,Test rows,Macro F1,Seed SD,Seeds
MiniLM + hierarchical GCN,1118,0.2701,0.0135,"13, 42, 87"
Character TF-IDF + logistic regression,1118,0.2421,—,42
LSA + logistic regression,1118,0.3593,—,42
MiniLM + hierarchical GCN · random-labels,1118,0.2817,0.0257,"13, 42, 87"
MiniLM + hierarchical GCN · self-loops,1118,0.2614,0.0062,"13, 42, 87"
MiniLM + hierarchical GCN · shuffled-edges,1118,0.2697,0.0418,"13, 42, 87"
MiniLM + logistic regression,1118,0.3451,—,42


## 4. Multilingual extension and source-only transfer

Read scores within each native label space. The Slavic directions vary both language
and genre. The Hindi and Portuguese pilots use binary targets. The same-language
transfer rows show the corresponding source-only controls.

In [5]:
for name, matrix in multilingual["matrices"].items():
    print(matrix["dataset"])
    models = [r for r in study("Native multilingual extension") if r["dataset"] == matrix["dataset"]]
    runs = [dict(r, seed_std=r["details"]["sample_std"], seeds=r["details"]["seeds"]) for r in models]
    if "training_majority_baseline" in matrix:
        baseline = matrix["training_majority_baseline"]
        runs.append(dict(name="Training-majority baseline", n=baseline["count"], value=baseline["macro_f1"]))
    classification(runs)

print("ProBel English/Arabic source-only transfer")
classification(study("ProBel news transfer pilot · initial study"))
print("Slavic Bulgarian/Russian source-only transfer")
table(("Method", "Train language", "Test language", "Test rows", "Macro F1"), (
    (r["method"], r["source_language"], r["target_language"], r["metrics"]["count"],
     score(r["metrics"]["macro_f1"]))
    for r in multilingual["transfer"]["runs"]
))

print("YASO targeted sentiment control")
classification(study("YASO clear-text subset · initial study"))

Slavic BG/RU sentence persuasion grouped pilot
H-Prop-News native Hindi grouped pilot
PartisanLens Portugal-publisher headline pilot
ProBel English/Arabic source-only transfer
Slavic Bulgarian/Russian source-only transfer
YASO targeted sentiment control


Model or control,Test rows,Macro F1,Seed SD,Seeds
TF-IDF + LR,300,0.0323,—,42
LSA + LR,300,0.0745,—,42
MiniLM + LR,300,0.1329,—,42
MiniLM + GCN,300,0.0502,0.0235,"13, 42, 87"
MiniLM + GCN / self-loops,300,0.0371,0.0044,"13, 42, 87"
MiniLM + GCN / shuffled-edges,300,0.0378,0.0191,"13, 42, 87"
MiniLM + GCN / random-labels,300,0.0503,0.0170,"13, 42, 87"


Model or control,Test rows,Macro F1,Seed SD,Seeds
TF-IDF + LR,150,0.8321,—,42
LSA + LR,150,0.8186,—,42
MiniLM + LR,150,0.7994,—,42
Training-majority baseline,150,0.3506,—,—


Model or control,Test rows,Macro F1,Seed SD,Seeds
TF-IDF + LR,33,0.6182,—,42
LSA + LR,33,0.6700,—,42
MiniLM + LR,33,0.7843,—,42
Training-majority baseline,33,0.4310,—,—


Model or control,Test rows,Macro F1,Seed SD,Seeds
ar-to-ar: graph,200,0.0670,—,42
ar-to-en: graph,200,0.0606,—,42
en-to-ar: graph,200,0.0187,—,42
en-to-en: graph,200,0.0371,—,42
ar-to-ar: transformer,200,0.0808,—,42
ar-to-en: transformer,200,0.0343,—,42
en-to-ar: transformer,200,0.0401,—,42
en-to-en: transformer,200,0.0337,—,42


Method,Train language,Test language,Test rows,Macro F1
transformer,bg,bg,150,0.0979
transformer,bg,ru,150,0.0460
graph,bg,bg,150,0.0529
graph,bg,ru,150,0.0514
transformer,ru,bg,150,0.0243
transformer,ru,ru,150,0.0538
graph,ru,bg,150,0.0470
graph,ru,ru,150,0.0128


Model or control,Test rows,Macro F1,Seed SD,Seeds
Character TF-IDF + logistic regression,305,0.2945,—,42
LSA + logistic regression,305,0.4390,—,42
MiniLM + logistic regression,305,0.4082,—,42


## 5. Subtask 3 · Evidence and explanation

The longer-text CARDS pilot separates supplied and predicted targets and compares
different sources of evidence. Its saved format rates include extractive fallbacks.
The two grounding diagnostics measure source support through different procedures.
The blinded model audit is archived and its exact judge runtime identifier was
unavailable. These diagnostics provide neither human ratings nor a truth assessment.

Reference similarity is reported separately for ProBel and e-SNLI. ProBel references
were model-authored and assessed by the dataset publishers. e-SNLI uses human
references and includes a copy-input control.

In [6]:
long_runs = study("CARDS · longer-text explanations")
stats = long_runs[0]["details"]["study_statistics"]
print(f'{stats["source_passages"]} passages, '
      f'{percent(1 - stats["identical_evidence_rate"])} differed in semantic vs. lead evidence, '
      f'{percent(stats["predicted_claim_label_accuracy"])} predicted target accuracy.')
table(("Method", "Passages", "Length and fallback"), (
    (r["name"], r["n"], r["format"]) for r in long_runs
))
table(("Grounding diagnostic", "Outputs", "Metric", "Value"), (
    (r["name"], r["n"], r["metric"], percent(r["value"]))
    for r in study("CARDS · independent grounding review")
))
reference_studies = ("ProBel news explanations pilot · initial study", "e-SNLI · initial study")
for name in reference_studies:
    print(name)
    table(("Method", "Cases", "BERTScore F1"), (
        (r["name"], r["n"], score(r["value"])) for r in study(name)
    ))

24 passages, 83.3% differed in semantic vs. lead evidence, 37.5% predicted target accuracy.
ProBel news explanations pilot · initial study
e-SNLI · initial study


Method,Passages,Length and fallback
predicted-semantic,24,Length compliance 100%; fallback 17%
semantic-extract,24,Length compliance 100%; fallback 0%
supplied-lead,24,Length compliance 100%; fallback 12%
supplied-semantic,24,Length compliance 100%; fallback 21%
textrank,24,Length compliance 100%; fallback 0%


Grounding diagnostic,Outputs,Metric,Value
predicted-semantic · NLI diagnostic,6,nli_source_support_rate,22.2%
predicted-semantic · blinded LLM audit,6,llm_fully_supported_rate,16.7%
semantic-extract · NLI diagnostic,6,nli_source_support_rate,44.1%
semantic-extract · blinded LLM audit,6,llm_fully_supported_rate,100.0%
supplied-lead · NLI diagnostic,6,nli_source_support_rate,27.8%
supplied-lead · blinded LLM audit,6,llm_fully_supported_rate,16.7%
supplied-semantic · NLI diagnostic,6,nli_source_support_rate,44.8%
supplied-semantic · blinded LLM audit,6,llm_fully_supported_rate,16.7%
textrank · NLI diagnostic,6,nli_source_support_rate,57.2%
textrank · blinded LLM audit,6,llm_fully_supported_rate,100.0%


Method,Cases,BERTScore F1
Qwen · leading evidence,40,0.7098
Qwen · semantic evidence,40,0.7098
Semantic extraction,40,0.6836
TextRank extraction,40,0.6836


Method,Cases,BERTScore F1
Copied input,60,0.8058
Label-only template,60,0.6935
Qwen · full context,60,0.7331
Qwen · premise withheld,60,0.7121


### Controlled three-generator pilot

All three models received the same prompts and selected evidence, using the
native target labels. The raw validity check includes the 80-word cap. The
named model judge assessed the same six passages per generator, including
raw responses that failed the format check. No paired human ratings were available.

In [7]:
generators = saved("generator-comparison")
review = saved("named-review")
by_model = {r["model"]: r for r in review["runs"]}
assert generators["complete"]
print(f'Judge: {review["judge"]["name"]}; paired human items: {review["human_paired_items"]}.')
table(("Generator", "Valid raw / total", "Median request (s)", "Judge mean / 2", "Reviewed"), (
    (r["name"], f'{round(r["valid_generation_rate"] * r["count"])}/{r["count"]}',
     f'{r["median_seconds"]:.2f}', f'{by_model[r["model"]]["llm_overall_mean"]:.2f}',
     by_model[r["model"]]["reviewed_outputs"])
    for r in generators["runs"]
))
table(("Generator", "Source support", "Attribution", "Relevance", "Readability"), (
    (r["name"], *(f'{by_model[r["model"]]["llm_mean_ratings"][k]:.2f}'
                  for k in ("supported", "attribution", "relevance", "readability")))
    for r in generators["runs"]
))

Judge: Qwen2.5-3B-Instruct; paired human items: 0.


Generator,Valid raw / total,Median request (s),Judge mean / 2,Reviewed
Qwen3-1.7B,17/24,7.49,1.17,6
SmolLM2-1.7B-Instruct,18/24,8.33,1.08,6
TinyLlama-1.1B-Chat-v1.0,0/24,9.72,1.29,6


Generator,Source support,Attribution,Relevance,Readability
Qwen3-1.7B,1.00,0.33,2.00,1.33
SmolLM2-1.7B-Instruct,1.00,0.33,1.67,1.33
TinyLlama-1.1B-Chat-v1.0,1.17,0.67,2.00,1.33


## 6. Processing and inference measurements

The DuckDB query aggregates record and positive counts by language for matching JSONL
and Parquet workloads. First scans can include initialization and filesystem cache
effects. Repeat cached prediction reads precomputed embeddings and includes scoring
and decoding. It excludes transformer encoding and generation. Process memory is
reported in MiB.

In [8]:
storage = [dict(r["details"], count=r["n"]) for r in study("Data processing scalability")]
conversion = {r["count"]: r for r in storage if r["operation"] == "convert"}
scans = {(r["count"], r["operation"]): r for r in storage if r["operation"] != "convert"}
table(("Rows", "Format", "Size (MB)", "First scan (s)", "Repeat (ms)", "Peak MiB"), (
    (n, label, f'{conversion[n][operation + "_bytes"] / 1e6:.3f}',
     f'{scans[n, operation]["first_scan_seconds"]:.3f}',
     f'{1000 * scans[n, operation]["repeat_scan_seconds"]:.2f}',
     f'{scans[n, operation]["peak_rss_mb"]:.1f}')
    for n in sorted(conversion) for operation, label in (("json", "JSONL"), ("parquet", "Parquet"))
))
table(("Rows", "JSONL to Parquet conversion (s)", "Conversion peak MiB"), (
    (n, f'{r["seconds"]:.3f}', f'{r["peak_rss_mb"]:.1f}')
    for n, r in sorted(conversion.items())
))

embeddings = {r["n"]: r["details"] for r in study("Embedding cache scalability")}
table(("Rows", "Encoder load (s)", "Uncached rows/s", "Cached rows/s", "Vectors identical", "Peak MiB"), (
    (n, f'{r["model_load_seconds"]:.2f}', f'{r["uncached_records_per_second"]:.2f}',
     f'{r["cached_records_per_second"]:.2f}', r["vectors_identical"], f'{r["peak_rss_mb"]:.1f}')
    for n, r in sorted(embeddings.items())
))
table(("Classifier", "Rows", "Checkpoint load (ms)", "First prediction (s)", "Repeat rows/s", "Peak MiB"), (
    (r["model"], r["count"], f'{1000 * r["model_load_seconds"]:.1f}',
     f'{r["first_inference_seconds"]:.3f}', f'{r["repeat_records_per_second"]:,.1f}', f'{r["peak_rss_mb"]:.1f}')
    for entry in study("Core model inference throughput") for r in [entry["details"]]
))

Rows,Format,Size (MB),First scan (s),Repeat (ms),Peak MiB
1000,JSONL,0.304,1.319,10.01,102.2
1000,Parquet,0.103,1.199,4.04,101.6
5000,JSONL,1.491,1.148,23.10,105.2
5000,Parquet,0.457,1.195,3.11,101.8
10000,JSONL,2.994,1.186,39.34,106.8
10000,Parquet,0.904,1.266,3.60,101.8
25000,JSONL,7.488,1.215,74.52,111.6
25000,Parquet,2.238,1.480,2.47,101.9


Rows,JSONL to Parquet conversion (s),Conversion peak MiB
1000,0.075,42.8
5000,0.096,45.9
10000,0.129,48.8
25000,0.258,58.4


Rows,Encoder load (s),Uncached rows/s,Cached rows/s,Vectors identical,Peak MiB
32,13.33,14.93,51.28,True,759.0
128,12.89,20.12,59.68,True,769.9


Classifier,Rows,Checkpoint load (ms),First prediction (s),Repeat rows/s,Peak MiB
minilm + hierarchical GCN,32,15.4,0.439,"1,930.6",299.0
minilm + hierarchical GCN,128,18.5,1.545,"2,017.3",300.0
minilm + hierarchical GCN,400,20.6,4.424,"1,960.3",303.8
xlmr + hierarchical GCN,32,15.5,0.357,"1,158.1",300.3
xlmr + hierarchical GCN,128,21.7,0.814,"1,283.6",301.0
xlmr + hierarchical GCN,400,21.7,2.368,"1,427.7",305.9


## 7. Run provenance

Source revisions identify when the underlying experiments were recorded. The [results guide](../results/README.md) lead to the
original aggregate files and protocols. A fresh run may produce different timings
and may retain archived studies at their original revisions.

In [9]:
included_studies = (
    *main_studies, "Native multilingual extension", "ProBel news transfer pilot · initial study",
    "Slavic source-only transfer", "YASO clear-text subset · initial study",
    "CARDS · longer-text explanations", "CARDS · independent grounding review", *reference_studies,
    "Data processing scalability", "Embedding cache scalability", "Core model inference throughput",
)
sources = [
    ("Main Spark audit", "spark-audit.json", spark["commit"]),
    ("Native multilingual Spark audit", "multilingual.json", multilingual["spark_audit"]["commit"]),
    ("Entity framing", "entity-framing.json", entity["commit"]),
    ("Three-generator pilot", "generator-comparison.json", generators["commit"]),
    ("Named model review", "named-review.json", review["commit"]),
]
for name in included_studies:
    commits = sorted({r["provenance"]["commit"] for r in study(name)})
    sources.extend((name, "experiments.json", commit) for commit in commits)
table(("Study", "Aggregate file in results/", "Source commit"), sources)

Study,Aggregate file in results/,Source commit
Main Spark audit,spark-audit.json,8bfa031a0814b8d9cd480d8e52fe1d7a03dd53e8
Native multilingual Spark audit,multilingual.json,6fda4021192bb7cbd1dffb50ce91c971653d4da2
Entity framing,entity-framing.json,399ac35e9e4611d9b2d9e20581c7f52abc7dbb23
Three-generator pilot,generator-comparison.json,4c71d9c3de45ec5107ccfed09f4a252fdb151f75
Named model review,named-review.json,667f8871d8cc89489f5cd2a22d3e40e5364951bc
CARDS · extended hierarchy study,experiments.json,8bfa031a0814b8d9cd480d8e52fe1d7a03dd53e8
ProBel · extended bilingual study,experiments.json,8bfa031a0814b8d9cd480d8e52fe1d7a03dd53e8
CARDS → EMM source transfer,experiments.json,5bb7e966a4db97a7ca1bd4be8ab32cc3b9fe6445
Native multilingual extension,experiments.json,00f8eb2065fddbc33a41aa1e6d97168630dfc0b6
Native multilingual extension,experiments.json,6fda4021192bb7cbd1dffb50ce91c971653d4da2


## 8. Optional · Reproduce the experiment runs

Follow the [repository setup](../README.md#setup-and-reproduction) first. Use
the kernel from the project environment, install Java for Spark, and place the
pinned llama.cpp build at the path below. Model and dataset downloads require a
network connection. Prefer a fresh checkout or preserve existing local results if
the runner reports a source or input hash mismatch.

Select `"main"` for setup, data audit, the CARDS/ProBel models, EMM transfer,
longer-text explanations, generator comparison and named review, systems
measurements, and the repeatable grounding diagnostic. Select `"multilingual"`
for the native-language extension and Slavic transfer. Select `"entity"` after
those two workflows to build the silver entity pilot. The optional `"auxiliary"`
selection covers the native reference comparisons, ProBel language transfer and
YASO. For its generation runs, start the pinned Qwen3 server on port 8080 as
described in the [README](../README.md#6-inspect-results-test-the-code-and-generate-local-explanations).
Run the main workflow before starting that server.

The cell runs in the order shown. Fresh runs write their normal local artifacts
and may update the exports. The saved results above remain a record of the original
runs until you execute the display cells again. Auxiliary reruns save new local
summaries in artifacts/ and leave the historical catalog entries intact. The archived
blinded Codex review
requires a separate evaluation using [its protocol](../docs/llm-review-prompt.md).

In [10]:
import subprocess
import sys

# Keep this empty for the offline fallback. Choose one or more workflow names.
RERUN = ()
LLAMA_SERVER = "work/llama-b11960/llama-server.exe"
QWEN_MODEL = "work/models/Qwen3-1.7B-Q4_K_M.gguf"

workflows = {
    "main": [
        ("experiments.download_models",),
        ("experiments.reproduce", "--stage", "all", "--server", LLAMA_SERVER),
    ],
    "multilingual": [("experiments.multilingual", "--stage", "all")],
    "entity": [("experiments.entity_framing", "--stage", "all")],
    "auxiliary": [
        ("experiments.run", "data/public/yaso/experiment.json", "--resume"),
        ("experiments.probel_transfer",),
        ("experiments.probel_explanations", "--gguf", QWEN_MODEL),
        ("experiments.public_explanations", "--gguf", QWEN_MODEL),
    ],
}
unknown = set(RERUN) - workflows.keys()
if unknown:
    raise ValueError(f"Unknown workflows: {sorted(unknown)}")
if RERUN:
    for name, commands in workflows.items():
        if name in RERUN:
            for args in commands:
                print("Running", "-m", *args, flush=True)
                subprocess.run([sys.executable, "-m", *args], cwd=root, check=True)
    subprocess.run([sys.executable, "-m", "experiments.reproduce", "--stage", "export"], cwd=root, check=True)
else:
    print("Archived results displayed. Fresh experiment runs are disabled.")

Archived results displayed. Fresh experiment runs are disabled.
